# DroneGeo: Interactive Quickstart & End-to-End Survey Tutorial

Welcome to **`dronegeo`**, the modular high-performance Python toolkit for drone remote sensing, UAV LiDAR point cloud processing, true-color orthomosaics, photogrammetric vegetation indices, terrain morphology, hydrological flow modeling, and 3D earthwork volumetrics.

### In this interactive notebook, you will learn how to:
1. **Pre-Processing Quality Control**: Audit point density, pulse returns, classifications, and plot pre-flight dashboards.
2. **Multi-Strip Co-Registration**: Detect vertical datum shifts across overlapping flightlines.
3. **Survey-Grade Surface Models**: Generate continuous IDW DTMs, DSMs, and Canopy Height Models (CHM).
4. **True-Color Orthomosaics & Vegetation Indices**: Generate 4-band RGBA orthos and calculate VARI & GLI crop health maps.
5. **Terrain Morphology & Contours**: Generate analytical hillshades, slope, aspect, and export vector contours.
6. **Hydrological Flow Routing & Risk Modeling**: Extract flow directions (D8 & D-inf), streams, TWI, SPI, STI, and landslide hazard indices.
7. **3D Earthwork Volumetrics**: Compute excavation Cut & Fill volumes and stockpile footprints.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Import DroneGeo
import dronegeo as dg

print(f"DroneGeo Version: {dg.__version__}")

## 0. Prepare Standard Sample Datasets
We locate or generate sample LiDAR survey datasets into `examples/data/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo_root = Path("..") if Path("../examples").exists() else Path(".")
DATA_DIR = (repo_root / "examples" / "data").resolve()
OUTPUT_DIR = (Path(".") / "outputs_notebook").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gen_script = repo_root / "examples" / "00_generate_sample_datasets.py"
if not (DATA_DIR / "flight_survey_master.las").exists():
    subprocess.run([sys.executable, str(gen_script)], check=True)

MASTER_LAS = DATA_DIR / "flight_survey_master.las"
print(f"Master point cloud ready: {MASTER_LAS}")

## 1. Pre-Processing Point Cloud Audit & Quality Control

In [ ]:
report = dg.lidar.profile_point_cloud(str(MASTER_LAS))

print(f"Total Points        : {report.total_points:,}")
print(f"Mean Point Density  : {report.mean_point_density:.2f} pts/m2")
print(f"Classification Mix  : {report.classification_percentages}")

qc_png = OUTPUT_DIR / "point_cloud_qc.png"
dg.lidar.plot_point_cloud_profile(report, output_png=str(qc_png))

from IPython.display import Image
Image(filename=str(qc_png))

## 2. Multi-Strip Flightline Alignment (dZ Co-Registration)

In [ ]:
strip1 = str(DATA_DIR / "flight_strip_01.las")
strip2 = str(DATA_DIR / "flight_strip_02.las")

alignment_report = dg.diagnostics.check_strip_alignment(strip1, strip2, sample_resolution=0.5)

print(f"Detected Median Datum Shift (dZ): {alignment_report.median_offset:+.4f} m")
print(f"Standard Deviation              : {alignment_report.std_dev:.4f} m")

hist_png = OUTPUT_DIR / "overlap_residuals.png"
dg.profiling.plot_strip_overlap_residuals(alignment_report, str(hist_png))
Image(filename=str(hist_png))

## 3. High-Resolution Surface Models (DTM, DSM, and CHM)

In [ ]:
dtm_tif = str(OUTPUT_DIR / "dtm_025m.tif")
dsm_tif = str(OUTPUT_DIR / "dsm_025m.tif")
chm_tif = str(OUTPUT_DIR / "chm_025m.tif")

# 1. Continuous Ground DTM via k-NN IDW
dg.dem.create_dtm(str(MASTER_LAS), dtm_tif, resolution=0.25, k_neighbors=8, ground_class=2)

# 2. Maximum Return DSM
dg.dem.create_dsm(str(MASTER_LAS), dsm_tif, resolution=0.25)

# 3. Canopy Height Model (CHM = DSM - DTM)
dg.dem.create_chm(dsm_tif, dtm_tif, chm_tif, clamp_min=0.0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, path, title, cmap in zip(axes, [dtm_tif, dsm_tif, chm_tif], ["Bare Earth DTM", "Surface DSM", "Canopy Height (CHM)"], ["terrain", "terrain", "viridis"]):
    with rasterio.open(path) as src:
        data = src.read(1)
        data[data == src.nodata] = np.nan
        im = ax.imshow(data, cmap=cmap)
        ax.set_title(title, fontsize=13, fontweight="bold")
        plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

## 4. True-Color Orthomosaics & Agricultural Vegetation Indices (VARI & GLI)

In [ ]:
ortho_tif = str(OUTPUT_DIR / "true_color_ortho.tif")
vari_tif = str(OUTPUT_DIR / "crop_vari.tif")
gli_tif = str(OUTPUT_DIR / "leaf_gli.tif")

# 1. True-Color Orthomosaic
dg.imagery.create_true_color_orthomosaic(str(MASTER_LAS), ortho_tif, resolution=0.25, alpha_channel=True, auto_contrast=True)

# 2. Visible Vegetation Indices
dg.imagery.compute_vari(ortho_tif, vari_tif)
dg.imagery.compute_gli(ortho_tif, gli_tif)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
with rasterio.open(ortho_tif) as src:
    rgb = np.dstack([src.read(1), src.read(2), src.read(3)])
    axes[0].imshow(rgb)
    axes[0].set_title("True-Color Orthomosaic (RGB)", fontweight="bold")

for ax, path, title, cmap in zip(axes[1:], [vari_tif, gli_tif], ["VARI Index (Crop Greenness)", "GLI Index (Chlorophyll)"], ["RdYlGn", "YlGn"]):
    with rasterio.open(path) as src:
        data = src.read(1)
        data[data == src.nodata] = np.nan
        im = ax.imshow(data, cmap=cmap)
        ax.set_title(title, fontweight="bold")
        plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

## 5. Terrain Morphology: Hillshade, Slope, Aspect & Contours

In [ ]:
hs_tif = str(OUTPUT_DIR / "hillshade.tif")
slope_tif = str(OUTPUT_DIR / "slope.tif")
contours_file = str(OUTPUT_DIR / "contours_1m.geojson")

# Analytical 8-bit Hillshade
dg.analysis.generate_hillshade(dtm_tif, hs_tif, azimuth_deg=315.0, altitude_deg=45.0)

# Slope Map
dg.analysis.generate_slope_map(dtm_tif, slope_tif, units="degrees")

# 1.0m Vector Contours
contours_gdf = dg.analysis.generate_contour_lines(dtm_tif, contours_file, interval_m=1.0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
with rasterio.open(hs_tif) as src:
    axes[0].imshow(src.read(1), cmap="gray")
    axes[0].set_title("Analytical Hillshade (NW 315 deg)", fontweight="bold")

with rasterio.open(slope_tif) as src:
    im = axes[1].imshow(src.read(1), cmap="magma")
    axes[1].set_title("Topographic Slope (Degrees)", fontweight="bold")
    plt.colorbar(im, ax=axes[1], shrink=0.7)

with rasterio.open(hs_tif) as src:
    axes[2].imshow(src.read(1), cmap="gray")
contours_gdf.plot(ax=axes[2], column="elevation", cmap="plasma", linewidth=1.2)
axes[2].set_title("1.0m Vector Contours Overlay", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Hydrological Flow Routing & Terrain Risk Modeling

Algorithms based on peer-reviewed scientific literature:
- **D8 & D-Infinity Flow Routing** (*O'Callaghan & Mark 1984; Tarboton 1997*)
- **Topographic Wetness Index (TWI)** (*Beven & Kirkby 1979*)
- **Stream Power Index (SPI)** (*Moore et al. 1991*)
- **Sediment Transport Index (STI / USLE LS)** (*Moore & Burch 1986*)
- **Multi-Criteria Landslide Hazard Index** (*Montgomery & Dietrich 1994*)

In [ ]:
d8_tif = str(OUTPUT_DIR / "flow_d8.tif")
accum_tif = str(OUTPUT_DIR / "flow_accum.tif")
twi_tif = str(OUTPUT_DIR / "twi.tif")
hazard_tif = str(OUTPUT_DIR / "landslide_hazard.tif")

# Hydrology & Risk models
dg.hydrology.compute_d8_flow_direction(dtm_tif, d8_tif)
dg.hydrology.compute_flow_accumulation(dtm_tif, accum_tif, units="cells")
dg.hydrology.compute_topographic_wetness_index(dtm_tif, twi_tif)
dg.hydrology.compute_landslide_susceptibility_index(dtm_tif, hazard_tif)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
with rasterio.open(accum_tif) as src:
    acc = np.log1p(src.read(1))
    axes[0].imshow(acc, cmap="Blues")
    axes[0].set_title("Log Flow Accumulation (Drainage)", fontweight="bold")

with rasterio.open(twi_tif) as src:
    twi = src.read(1)
    twi[twi == src.nodata] = np.nan
    im = axes[1].imshow(twi, cmap="YlGnBu")
    axes[1].set_title("Topographic Wetness Index (TWI)", fontweight="bold")
    plt.colorbar(im, ax=axes[1], shrink=0.7)
    
with rasterio.open(hazard_tif) as src:
    haz = src.read(1).astype(float)
    haz[haz == 255] = np.nan
    im = axes[2].imshow(haz, cmap="RdYlGn_r")
    axes[2].set_title("Landslide Susceptibility Hazard Score [0-100]", fontweight="bold")
    plt.colorbar(im, ax=axes[2], shrink=0.7)
plt.tight_layout()
plt.show()

## 7. 3D Cut & Fill Earthwork Volumetrics

In [ ]:
epoch1 = str(DATA_DIR / "quarry_epoch1.tif")
epoch2 = str(DATA_DIR / "quarry_epoch2.tif")
diff_tif = str(OUTPUT_DIR / "quarry_diff.tif")

vol_report = dg.analysis.compute_cut_fill_volume(epoch1, epoch2, output_diff_tif=diff_tif)

print(f"Excavated Cut Volume : {vol_report.cut_volume_m3:,.2f} m3")
print(f"Deposited Fill Volume: {vol_report.fill_volume_m3:,.2f} m3")
print(f"Net Volume Balance   : {vol_report.net_volume_m3:+,.2f} m3")

with rasterio.open(diff_tif) as src:
    diff = src.read(1)
    diff[diff == src.nodata] = np.nan
    plt.figure(figsize=(8, 6))
    plt.imshow(diff, cmap="coolwarm", vmin=-6, vmax=6)
    plt.colorbar(label="Elevation Change dZ (m)")
    plt.title("3D Cut & Fill Elevation Delta (Epoch 1 vs Epoch 2)", fontweight="bold")
    plt.show()